# 00 · Setup & orientation

This `notebooks/` folder is a **step-by-step, traceable** view of the same benchmark
implemented in `benchmark/`. It follows a **hybrid** design:

- **Shared logic is imported** from `benchmark/` (constants, model list, grids,
  `evaluate()`) — one source of truth, so these notebooks can never drift from the
  package.
- **The pipeline transformation, orchestration, and plots are inlined** cell-by-cell,
  so the methodology is visible as you read.

It is **non-destructive**: notebooks *read* the existing `artifacts/` and
`results/results.csv`; they never overwrite them. Heavy training is gated behind
`RUN_*` flags (default `False`) — the model notebooks load precomputed results and
just visualize.

**Run order:** `00` → `01_data_pipeline` → `02_pass1_baselines` →
`03_pass2_gridsearch` → `04_analysis_findings`.

In [1]:
# --- make the benchmark/ package importable from notebooks/ ---
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'benchmark').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

project root: D:\_fcall\UNIFEI\tcc\hpc_workload


In [2]:
import benchmark.config as C
import numpy as np, json

print('SEED               =', C.SEED)
print('WINDOW_SIZE        =', C.WINDOW_SIZE, '(40 snapshots x 15 s = 10 min)')
print('PRE_FAILURE_HOURS  =', C.PRE_FAILURE_HOURS)
print('TRAIN_FRACTION     =', C.TRAIN_FRACTION)
print('FAILURE_STATES     =', C.FAILURE_STATES)
print('artifacts dir      =', C.ARTIFACTS_DIR)

SEED               = 42
WINDOW_SIZE        = 40 (40 snapshots x 15 s = 10 min)
PRE_FAILURE_HOURS  = 2
TRAIN_FRACTION     = 0.8
FAILURE_STATES     = {'TIMEOUT', 'OUT_OF_MEMORY', 'FAILED', 'NODE_FAIL'}
artifacts dir      = D:\_fcall\UNIFEI\tcc\hpc_workload\artifacts


## The 8 metrics → 32 windowed features

In [3]:
print('8 base metrics:')
for f in C.FEATURES:
    print('  -', f)
fn = C.feature_names()
print(f'\n-> {len(fn)} windowed features (each metric x min/max/mean/std). e.g.:')
print('  ', fn[:2], '...', fn[-2:])

8 base metrics:
  - node_memory_Active_bytes
  - node_disk_written_bytes_total_sum
  - node_netstat_Tcp_InErrs
  - node_forks_total
  - nvidia_gpu_temperature_celsius_mean
  - nvidia_gpu_power_usage_milliwatts_mean
  - nvidia_gpu_memory_used_bytes_sum
  - nvidia_gpu_fanspeed_percent_mean

-> 32 windowed features (each metric x min/max/mean/std). e.g.:
   ['node_memory_Active_bytes_min', 'node_disk_written_bytes_total_sum_min'] ... ['nvidia_gpu_memory_used_bytes_sum_std', 'nvidia_gpu_fanspeed_percent_mean_std']


## Artifacts produced by the pipeline (read-only here)

In [4]:
assert C.artifacts_exist(), 'Run  python -m benchmark.data_pipeline  first.'
for k in ['X_train','X_test','y_train','y_test','states_train','states_test','job_ids_train','job_ids_test']:
    a = np.load(C.artifact_path(k), mmap_mode='r')
    print(f'{k:16s} shape={str(a.shape):18s} dtype={a.dtype}')
print('\nstate code map:', C.load_state_names())

X_train          shape=(9125945, 32)      dtype=float32
X_test           shape=(1411809, 32)      dtype=float32


y_train          shape=(9125945,)         dtype=int8
y_test           shape=(1411809,)         dtype=int8
states_train     shape=(9125945,)         dtype=int8
states_test      shape=(1411809,)         dtype=int8
job_ids_train    shape=(9125945,)         dtype=int32


job_ids_test     shape=(1411809,)         dtype=int32

state code map: {0: 'COMPLETED', 1: 'TIMEOUT', 2: 'FAILED', 3: 'OUT_OF_MEMORY', 4: 'NODE_FAIL'}
